# WALINET lipid-mask degradation evaluation

For every subject and dilation level, this experiment recomputes the lipid-projection operator. The simulated input spectra and targets are shared across all dilation levels; only the YNet L2 input changes.

In [ ]:
from pathlib import Path
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from walinet.paper_figures.mask_degradation.pipeline import run_evaluation

config_path = Path("../configs/MaskDegeneration/mask_degradation.yaml")
if not config_path.is_file():
    config_path = Path("configs/MaskDegeneration/mask_degradation.yaml")
config_path = config_path.resolve()
result_path = run_evaluation(config_path, overwrite=False)
result_path

In [ ]:
from walinet.paper_figures.mask_degradation.results import (
    aggregates_dataframe,
    results_dataframe,
)

individual = results_dataframe(result_path)
summary = aggregates_dataframe(result_path)
summary

In [ ]:
import matplotlib.pyplot as plt

subject_means = (
    individual.groupby(["subject", "dilation", "metric"], as_index=False)["mean"]
    .mean()
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for axis, metric in zip(axes, ("nuisance_relative_l2", "metabolite_relative_l2")):
    for subject in subject_means["subject"].unique():
        subject_values = subject_means[(subject_means["metric"] == metric) &
                                       (subject_means["subject"] == subject)].sort_values("dilation")
        axis.plot(subject_values["dilation"], subject_values["mean"],
                  color="0.65", linewidth=1, alpha=0.8)
    values = summary[summary["metric"] == metric].sort_values("dilation")
    axis.errorbar(
        values["dilation"], values["mean"], yerr=values["std"],
        color="black", linewidth=2, marker="o", capsize=4,
        label="mean ± SD across 5 subjects"
    )
    axis.set_title(metric)
    axis.set_xlabel("Inward mask dilation (voxels)")
    axis.set_ylabel("Relative L2 error")
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()